# 4. Transcript inspection

## Objective

This notebook inspects the cached Whisper transcripts before extracting linguistic features. We check file shape, missing values, empty transcripts, transcript examples, and length statistics.

Transcript quality directly affects grammar features: missing or empty text produces zero or near-zero linguistic measurements. The later feature pipeline handles missing transcript values by replacing them with an empty string, so those rows remain aligned with the audio records instead of being dropped.

In [ ]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir("."))

In [ ]:
import pandas as pd
import os

PROJECT_ROOT = r"C:\Users\rudra\Desktop\SHL-Grammar-Scoring"

transcript_path = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "train_transcripts.csv"
)

print("Looking for:")
print(transcript_path)
print("Exists:", os.path.exists(transcript_path))

transcripts = pd.read_csv(transcript_path)

print("\nShape:", transcripts.shape)
print("\nColumns:", transcripts.columns.tolist())
print("\nMissing values:")
print(transcripts.isna().sum())

display(transcripts.head())

In [ ]:
empty = transcripts[
    transcripts["transcript"].isna() |
    (transcripts["transcript"].str.strip() == "")
]

print("Empty transcripts:", len(empty))
display(empty)

In [ ]:
import os
import librosa

PROJECT_ROOT = r"C:\Users\rudra\Desktop\SHL-Grammar-Scoring"

for filename in empty["filename"]:
    path = os.path.join(PROJECT_ROOT, "train", filename)

    y, sr = librosa.load(path, sr=None)

    print(
        filename,
        "| duration:", round(len(y) / sr, 2),
        "| max amplitude:", round(float(abs(y).max()), 4),
        "| RMS:", round(float((y ** 2).mean() ** 0.5), 4)
    )

## Interpretation

The inspection identifies the rows with missing or empty transcripts and provides length evidence for the remaining text. Missing transcripts are retained for alignment and later converted to empty strings before feature extraction. Consequently, their linguistic features represent unavailable text rather than silently dropping the corresponding audio and label.